## Conversion BDB → JSON

Conversion du dictionnaire BDB (Brown-Driver-Briggs Hebrew Lexicon) depuis le
fichier CSV d'Eliran Wong (`bdb.csv`) (https://github.com/eliranwong/unabridged-BDB-Hebrew-lexicon) vers un format JSON structuré.

### Source

- **Fichier** : `bdb.csv` (10 022 entrées, format TSV)
- **Colonnes** : `BDBid`, `StrongNumber`, `content` (HTML propriétaire)

### Structure JSON produite

| Champ | Type | Description |
|---|---|---|
| `dic` | str | Identifiant du dictionnaire |
| `sort_key` | int | Numéro d'ordre séquentiel |
| `strong` | str/null | Numéro Strong (`H24`), `null` si absent |
| `lang` | str | Langue : `"Hebrew"` ou `"Aramaic"` |
| `m` | list | Formes avec voyelles et accents |
| `b` | list | Formes nues sans diacritiques (pour la recherche) |
| `l` | list | Translitérations latines |
| `d` | str | Définition HTML (balise `<p>`) |

### Fonctions

- **`strip_vowels(text)`** — supprime les voyelles (niqqud)
- **`transliterate(text)`** — convertit en translitération latine via `TRANSLIT`
- **`extract_definition(content)`** — retourne tous les `<p>` d'une entrée CSV
- **`extract_headwords(p)`** — extrait les formes hébraïques/araméennes d'un `<p>`
- **`parse_row(rows)`** — convertit toutes les lignes CSV en entrées JSON


In [1]:
from bs4 import BeautifulSoup
import csv, sys, re, json


In [2]:
TRANSLIT = {
    # Consonnes
    '\u05D0': "'",
    '\u05D1': 'b',
    '\u05D2': 'g',
    '\u05D3': 'd',
    '\u05D4': 'h',
    '\u05D5': 'w',
    '\u05D6': 'z',
    '\u05D7': 'kh',
    '\u05D8': 't',
    '\u05D9': 'y',
    '\u05DB': 'k',
    '\u05DA': 'k',
    '\u05DC': 'l',
    '\u05DE': 'm',
    '\u05DD': 'm',
    '\u05E0': 'n',
    '\u05DF': 'n',
    '\u05E1': 's',
    '\u05E2': "'",
    '\u05E4': 'p',
    '\u05E3': 'p',
    '\u05E6': 'ts',
    '\u05E5': 'ts',
    '\u05E7': 'q',
    '\u05E8': 'r',
    '\u05E9': 'sh',
    '\u05EA': 't',
    # Voyelles (niqqud)
    '\u05B0': 'e', '\u05B1': 'e', '\u05B2': 'a', '\u05B3': 'o',
    '\u05B4': 'i', '\u05B5': 'e', '\u05B6': 'e', '\u05B7': 'a',
    '\u05B8': 'a', '\u05B9': 'o', '\u05BA': 'o', '\u05BB': 'u',
    '\u05BC': '',  # dagesh (ignoré)
    '\u05BE': '-', # maqqef (tiret)
    '\u05C1': '',  # shin dot
    '\u05C2': '',  # sin dot
}

In [3]:
def strip_vowels(text: str) -> str:
    """Supprime les voyelles (niqqud) d'une forme hébraïque."""
    return re.sub(r'[\u05B0-\u05C7\u05F0-\u05F4\uFB1D-\uFB4E]', '', text)


def transliterate(text: str, drop_gutturals: bool = False) -> str:
    """Convertit une forme hébraïque en translitération latine via TRANSLIT.

    Si drop_gutturals=True, ignore aleph (א) et ayin (ע).
    """
    text = text.replace('שׁ', 'sh').replace('שׂ', 's')
    result = ''
    for char in text:
        if drop_gutturals and char in ('א', 'ע'):
            result += ''
        else:
            result += TRANSLIT.get(char, char)
    return result


def extract_definition(content: str) -> list:
    """Retourne la liste de tous les <p> d'une entrée CSV. Liste vide si aucun <p>."""
    content_soup = BeautifulSoup(content, features='html.parser')
    return content_soup.find_all('p')


def extract_headwords(p) -> list[str]:
    """Extrait les formes hébraïques ou araméennes en tête d'un <p>.

    Prend le premier <bdbheb> ou <bdbarc>, puis suit les variantes
    séparées par des virgules si elles sont des siblings directs.
    """
    words = []
    entry = p.find(['bdbheb', 'bdbarc'])
    while entry:
        words.append(entry.text)
        nxt = entry.next_sibling
        if nxt is None or not str(nxt).strip().startswith(','):
            break
        candidate = nxt.next_sibling
        if candidate and candidate.name in ['bdbheb', 'bdbarc']:
            entry = candidate
        else:
            break
    return words


def parse_row(rows: list[dict]) -> list[dict]:
    """Convertit toutes les lignes du CSV en entrées JSON structurées.

    Chaque <p> contenant un headword hébreu ou araméen devient une entrée
    distincte. Les paragraphes sans headword sont ignorés.
    """
    sort_key = 1
    bdb = []
    for row in rows:
        for p in extract_definition(row['content']):
            if p.find(['bdbheb', 'bdbarc']) is None:
                continue
            m = extract_headwords(p)
            b = [strip_vowels(w) for w in m]
            bdb.append({
                "dic": "BDB",
                "sort_key": sort_key,
                "strong": row['StrongNumber'] or None,
                "lang": "Aramaic" if p.find('bdbarc') else "Hebrew",
                "m": m,
                "b": b,
                "l": [t for w in m for t in (transliterate(w), transliterate(w, drop_gutturals=True))] + [transliterate(w) for w in b],
                "d": str(p),
            })
            sort_key += 1
    return bdb


In [4]:
csv.field_size_limit(sys.maxsize)

with open('bdb.csv', encoding='utf-8') as f:
    rows = list(csv.DictReader(f, delimiter='\t'))

bdb = parse_row(rows)

with open('bdb.json', 'w', encoding='utf-8') as f:
    json.dump(bdb, f, ensure_ascii=False, indent=2)

print(f"Fichier bdb.json créé — {len(bdb)} entrées")


Fichier bdb.json créé — 11729 entrées
